# Imports

In [ ]:
from harbor.analysis import cross_docking as cd
import pandas as pd
from importlib import reload
reload(cd)

In [ ]:
raw_df = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/20250212_p_to_x_posit/20250311_combined_results") 

In [ ]:
settings = cd.Settings()

In [ ]:
settings.use_scaffold_split = True
settings.use_date_split = True
settings.use_random_split = False
settings.scaffold_split_option = cd.ScaffoldSplitOptions.X_TO_ALL
settings.query_scaffold_min_count = 5
settings.n_per_split = list(range(1, 11)) + list(range(15, len(df.Reference_Structure.unique())+20, 20))

In [ ]:
settings.to_yml_file("scaffold_split_settings.yml")

In [ ]:
cluster_sizes = raw_df.groupby(settings.query_scaffold_id_column)[
    settings.query_ligand_column
].nunique()

query_subset_list = [
    [scaffold]
    for scaffold in cluster_sizes[
        cluster_sizes > settings.query_scaffold_min_count
    ].index.tolist()
]

In [ ]:
date_dict_list = (
                raw_df.groupby(settings.reference_structure_column)[
                    [
                        settings.reference_structure_column,
                        settings.reference_structure_date_column,
                    ]
                ]
                .head(1)
                .to_dict(orient="records")
            )

simplified_date_dict = {
    date_dict[settings.reference_structure_column]: date_dict[
        settings.reference_structure_date_column
    ]
    for date_dict in date_dict_list
}

In [ ]:
settings.n_per_split = [1,5,20,100,315]
evs = [cd.Evaluator(dataset_split=cd.ScaffoldSplit(query_scaffold_id_column=settings.query_scaffold_id_column, 
                                                   reference_scaffold_id_column=settings.reference_scaffold_id_column, 
                                                   query_scaffold_id_subset=query_subset, 
                                                   split_option = settings.scaffold_split_option,
                                                   n_per_split=-1,),
                    extra_splits=[cd.DateSplit(reference_structure_column=settings.reference_structure_column, 
                                              n_per_split=n_per_split,
                                              balanced=True,
                                              date_dict=simplified_date_dict,
                                              randomize_by_n_days=settings.randomize_by_n_days,
                                              )],
                    scorer=scorer,
                    evaluator=cd.BinaryEvaluation(variable="RMSD", cutoff=2.0),
                    groupby=[settings.query_ligand_column]
                    
                    )   
       for query_subset in query_subset_list
       for n_per_split in settings.n_per_split
       for scorer in [cd.POSITScorer()]
       ]

In [ ]:
import tqdm
results = []
for ev in tqdm.tqdm(evs):
    results.append(cd.Results(evaluator=ev, fraction_good=ev.run(raw_df)))

In [ ]:
df = cd.Results.df_from_results(results)

In [ ]:
df

In [ ]:
df['Scaffold'] = df.Query_Scaffold_ID_Subset.apply(lambda x: x[0])

In [ ]:
import plotly.express as px

In [ ]:
fig = px.line(df, x="N_Per_Split_1", 
              y="Fraction", 
              color="Scaffold", 
              facet_col="Score", 
              template='simple_white',
              log_x=True,
              height=400,
              width=400)

In [ ]:
fig.show()

In [ ]:
dfs = []
testdf = raw_df.copy()
testdf = testdf[testdf.Pose_ID == 0]
testdf.sort_values('Reference_Structure_Date', inplace=True)
structure_list = testdf.groupby('Reference_Structure').head(1).Reference_Structure.to_list()
structure_lists = [structure_list[:i] for i in range(1, len(structure_list)+1, 10)]
lengths = [len(structure_list) for structure_list in structure_lists]
subset_list = [x[0] for x in query_subset_list]
print('running over clusters')
for cluster_id in subset_list:
    print(cluster_id)
    clusterdf = testdf[testdf.cluster_id == cluster_id]
    by_n_per_split = []
    for structure_list in structure_lists:
        nps = clusterdf[clusterdf.Reference_Structure.isin(structure_list)]
        nps.sort_values("docking-confidence-POSIT", ascending=False, inplace=True)
        final = nps.groupby("Query_Ligand").head(1)
        by_n_per_split.append(sum(final.RMSD < 2.0) / len(final))
    dfs.append(pd.DataFrame({'N_Per_Split': lengths, 'Fraction': by_n_per_split, 'Scaffold': cluster_id}))

In [ ]:
final_test_df = pd.concat(dfs)

In [ ]:
fig = px.line(final_test_df, x="N_Per_Split", y="Fraction", color="Scaffold", log_x=True, template='simple_white', height=400, width=400)

In [ ]:
fig.show()

# Don't dock to the same scaffold!

In [ ]:
dfs = []
testdf = raw_df.copy()
testdf = testdf[testdf.Pose_ID == 0]
testdf.sort_values('Reference_Structure_Date', inplace=True)
structure_list = testdf.groupby('Reference_Structure').head(1).Reference_Structure.to_list()
structure_lists = [structure_list[:i] for i in range(1, len(structure_list)+1, 10)]
lengths = [len(structure_list) for structure_list in structure_lists]
subset_list = [x[0] for x in query_subset_list]
print('running over clusters')
for cluster_id in subset_list:
    print(cluster_id)
    # this line is the only difference
    clusterdf = testdf[(testdf.cluster_id == cluster_id)&(testdf.cluster_id_Reference != cluster_id)]
    by_n_per_split = []
    for structure_list in structure_lists:
        nps = clusterdf[clusterdf.Reference_Structure.isin(structure_list)]
        nps.sort_values("docking-confidence-POSIT", ascending=False, inplace=True)
        final = nps.groupby("Query_Ligand").head(1)
        by_n_per_split.append(sum(final.RMSD < 2.0) / len(final))
    dfs.append(pd.DataFrame({'N_Per_Split': lengths, 'Fraction': by_n_per_split, 'Scaffold': cluster_id}))

In [ ]:
final_test_df = pd.concat(dfs)

In [ ]:
fig = px.line(final_test_df, x="N_Per_Split", y="Fraction", color="Scaffold", log_x=True, template='simple_white', height=600, width=600)

In [ ]:
reload(cd)
settings = cd.Settings()

In [ ]:
settings.use_scaffold_split = True

In [ ]:
settings.update_n_per_split(raw_df)

In [ ]:
dsplits = settings.create_dataset_splits(raw_df)

In [ ]:
len(dsplits)

In [ ]:
len(dsplits)

# combine dataset splits

In [ ]:
from collections import defaultdict
split_dict = defaultdict(list)
for ds in dsplits:
    split_dict[ds.name].append(ds)

In [ ]:
split_dict

In [ ]:
from itertools import product
combined_splits = []
for split1 in ['RandomSplit', 'DateSplit']:
    for split2 in ['SimilaritySplit', 'ScaffoldSplit']:
        split1s = split_dict[split1]
        split2s = split_dict[split2]
        for s1, s2 in product(split1s, split2s):
            combined_splits.append((s1, s2))

In [ ]:
len(combined_splits)

In [ ]:
combined_splits[0]

In [ ]:
reload(cd)
settings = cd.Settings()

In [ ]:
settings.use_scaffold_split = True
settings.scaffold_split_option = 'x_to_not_x'
settings.query_scaffold_min_count = 5
settings.combine_core_and_chemical_splits = True

In [ ]:
settings.update_n_per_split(raw_df)

In [ ]:
dsplits = settings.create_dataset_splits(raw_df)

In [ ]:
dataset_splits, extra_splits = settings.combine_splits(dsplits)

In [ ]:
len(dataset_splits)

In [ ]:
len(extra_splits)

In [ ]:
evs = settings.create_evaluators(raw_df)

In [ ]:
len(evs)

In [ ]:
s2 = cd.Settings()

In [ ]:
settings.similarity_range

In [ ]:
s2.similarity_range

In [ ]:
s2.similarity_range = (0,20)

In [ ]:
settings.similarity_range

In [ ]:
settings.to_yaml_file("scaffold_split_settings.yml")

In [ ]:
raw_df.Reference_Structure.nunique()

In [ ]:
evs[-1]

In [ ]:
evs[-1].run(raw_df)

In [ ]:
testdf = raw_df.copy()

In [ ]:
ev = evs[-1]
a = ev.pose_selector.run(testdf)

In [ ]:
b = ev.dataset_split.run(a)[0]

In [ ]:
c = ev.extra_splits[0].run(b)[0]

In [ ]:
d = ev.structure_choice.run()

In [ ]:
settings.scaffold_split_option in [cd.ScaffoldSplitOptions.X_TO_ALL, cd.ScaffoldSplitOptions.X_TO_NOT_X]

In [ ]:
scaffold_split = evs[-1].extra_splits[0]

In [ ]:
reload(cd)
ssdict = scaffold_split.model_dump_json()

In [ ]:
reload(cd)
ss = cd.ScaffoldSplit(**scaffold_split.dict())

In [ ]:
ss.run(raw_df)